In [7]:
import pandas as pd
import plotly.graph_objects as go

shared_layout = {
    "font": {
        "size": 16,
    },
    "margin": {
        "t": 25,
        "r": 20,
        "b": 70,
        "l": 35,
    },
    "legend_title": "Legend",
    "legend": {
        "orientation": "v",
        "x": 0.05,
        "xanchor": "left",
        "y": -0.08,
        "yanchor": "top",
        "font": {
            "size": 14,
        },
    },
}

comparison = pd.concat(
    [
        pd.read_csv("awe_plot_summary.csv"),
        pd.read_csv("pemwe_plot_summary.csv"),
    ],
    ignore_index=True,
)

colors = {
    "AWE": "#1f77b4",
    "PEMWE": "#ff7f0e",
}

def comparison_plot(metric, yaxis_title, filename=None):
    data = comparison.loc[comparison["Metric"] == metric]

    fig = go.Figure()

    for technology in ["AWE", "PEMWE"]:
        rows = (
            data.loc[data["Technology"] == technology]
            .set_index("Scenario")
            .loc[["S1", "S2"]]
        )

        fig.add_bar(
            name=technology,
            x=["S1", "S2"],
            y=rows["Center"],
            marker_color=colors[technology],
            error_y=dict(
                type="data",
                symmetric=False,
                array=(rows["Upper"] - rows["Center"]).to_numpy(),
                arrayminus=(rows["Center"] - rows["Lower"]).to_numpy(),
            ),
        )

    fig.update_layout(
        **shared_layout,
        title=None,
        barmode="group",
        xaxis_title="Scenario",
        yaxis_title=yaxis_title,
        legend_title_text="Technology",
        # template="plotly_white",
    )
    if filename:
        fig.write_image(f"{filename}.svg")

    return fig

comparison_plot(
    "LCOH",
    "LCOH [EUR/kg H₂]",
    filename="plots/lcoh_awe_pemwe_comparison.svg"
).show()

comparison_plot(
    "OPEX",
    "Lifecycle OPEX [EUR]",
    filename="plots/opex_awe_pemwe_comparison.svg"
).show()